In [1]:
!pip install nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 11.4 MB/s eta 0:00:0000:01


In [2]:
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 25.2 MB/s eta 0:00:0000:01


In [ ]:
import pandas as pd
import nlpaug.augmenter.word as naw
from sentence_transformers import SentenceTransformer
from transformers import MarianMTModel, MarianTokenizer
import torch
import numpy as np
import re
import random

In [ ]:
pd.set_option("display.max_colwidth", None)

In [ ]:
df = pd.read_csv('/kaggle/input/liar2-dataset/train.csv', encoding="utf-8")
df = df[['statement', 'label']]

In [ ]:
def analyze_data(df):
    print(df.head())
    print(df.describe())
    print(df.isnull().sum())
    print(df.shape)
    empty_string = (df == "").sum()
    print(empty_string)
    print(df['label'].value_counts())

In [ ]:
analyze_data(df)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
def clean(text):
    if pd.isna(text):
        return text
    text = text.lower()
    text = text.strip()
    text = re.sub(r'https\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [ ]:
df['statement'] = df['statement'].apply(lambda x: clean(x))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
def back_translate_batch(texts, to_lang, batch_size=16):
    from_model = f"Helsinki-NLP/opus-mt-en-{to_lang}"
    to_model = f"Helsinki-NLP/opus-mt-{to_lang}-en"

    tokenizer_f = MarianTokenizer.from_pretrained(from_model)
    model_f = MarianMTModel.from_pretrained(from_model).to(device)
    tokenizer_b = MarianTokenizer.from_pretrained(to_model)
    model_b = MarianMTModel.from_pretrained(to_model).to(device)

    augmented_texts = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer_f(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        translated = model_f.generate(**inputs)
        translated_text = tokenizer_f.batch_decode(translated, skip_special_tokens=True)

        inputs_back = tokenizer_b(translated_text, return_tensors="pt", padding=True, truncation=True).to(device)
        back_translated = model_b.generate(**inputs_back)
        back_translated_text = tokenizer_b.batch_decode(back_translated, skip_special_tokens=True)

        augmented_texts.extend(back_translated_text)

    return augmented_texts

In [ ]:
augmented_texts = back_translate_batch(
    df['statement'].tolist(),
    to_lang="de",
    batch_size=16
)

df['statement_aug_de'] = augmented_texts

In [ ]:
augmented_texts = back_translate_batch(
    df['statement'].tolist(),
    to_lang="et",
    batch_size=16
)

df['statement_aug_et'] = augmented_texts

In [ ]:
augmented_texts = back_translate_batch(
    df['statement'].tolist(),
    to_lang="ar",
    batch_size=16
)

df['statement_aug_ar'] = augmented_texts

In [ ]:
language_list = ['ca', 'fr']
for language in language_list:
    augmented_texts = back_translate_batch(
    df['statement'].tolist(),
    to_lang=language,
    batch_size=16
    )
    column_name = f'statement_aug_{language}'
    df[column_name] = augmented_texts

In [ ]:
df.columns

In [ ]:
df.to_csv('/kaggle/working/augmented_dataset.csv', index=False)